# Fine-Tuning with Kubeflow Trainer v2

This tutorial walks through how to use **Kubeflow Trainer v2** to run supervised fine-tuning (SFT) jobs with [LlamaFactory](https://github.com/hiyouga/LLaMA-Factory) on Kubernetes.

## Overview

Kubeflow Trainer v2 separates **job templates** (`TrainingRuntime`) from **job runs** (`TrainJob`), which lets you:

- Define a reusable `TrainingRuntime` that captures the container image, training pipeline steps (dataset init → model init → trainer), and LlamaFactory configuration.
- Submit many `TrainJob` runs referencing the same runtime, overriding only what changes per experiment — base model, dataset URL, hyperparameters, or GPU resources.

### Workflow

```
[One-time setup]                       [Per experiment]
TrainingRuntime  ──────────────────►  TrainJob
 - container image                      - base model URL
 - init steps (dataset, model)          - dataset URL
 - LlamaFactory train config            - numNodes / GPU resources
 - LoRA merge & push logic              - output model repo
```

The training pipeline inside the runtime consists of three sequential steps:

1. **`dataset-initializer`** — clones the dataset from a Git repository into a shared PVC.
2. **`model-initializer`** — clones the base model (with Git LFS) into the same PVC.
3. **`trainer`** — runs `llamafactory-cli train`, optionally merges LoRA adapters, then pushes the result back to a Git model repository.

## Step 1: Create a TrainingRuntime

A `TrainingRuntime` is a **cluster-scoped or namespace-scoped** reusable job template. You create it once and reference it from every `TrainJob`.

The runtime below defines three general steps:

- **`dataset-initializer`** — clones a dataset Git repo into `/mnt/models`.
- **`model-initializer`** — runs after `dataset-initializer` completes; clones the base model (with Git LFS) into `/mnt/models`.
- **`trainer`** — runs after `model-initializer` completes; generates a LlamaFactory SFT config, trains the model, merges LoRA adapters, and pushes the merged model back to a Git repo.

> **Note:** Both the `DATASET_URL` and `BASE_MODEL_URL` in the runtime below serve as **default values**. You will override them per-run in the `TrainJob`.

Execute the following block to create `kf-trainingruntime.yaml`:

In [ ]:
%%writefile kf-trainingruntime.yaml
apiVersion: trainer.kubeflow.org/v1alpha1
kind: TrainingRuntime
metadata:
  name: llamafactory-finetune-runtime
  labels:
    trainer.kubeflow.org/framework: torch
spec:
  mlPolicy:
    numNodes: 1
    torch:
      numProcPerNode: auto
  template:
    spec:
      replicatedJobs:
        - name: dataset-initializer
          template:
            metadata:
              labels:
                trainer.kubeflow.org/trainjob-ancestor-step: dataset-initializer
            spec:
              template:
                spec:
                  securityContext:
                    runAsNonRoot: true
                    runAsUser: 65534
                    runAsGroup: 65534
                    fsGroup: 65534
                  containers:
                    - name: dataset-initializer
                      image: alaudadockerhub/fine_tune_with_llamafactory:v0.1.11
                      command:
                      - /bin/bash
                      - -c
                      - |
                        set -ex
                        cd /mnt/models
                        DATASET_NAME=$(basename ${DATASET_URL})
                        DATASET_URL_NO_HTTPS="${DATASET_URL//https:\/\/}"
                        gitauth="${GIT_USER}:${GIT_TOKEN}"
                        rm -rf ${DATASET_NAME}
                        rm -rf data
                        if [ -d ${DATASET_NAME} ]; then
                            echo "dataset ${DATASET_NAME} already exists skipping download"
                        else
                            git -c http.sslVerify=false -c lfs.activitytimeout=36000 clone "https://${gitauth}@${DATASET_URL_NO_HTTPS}"
                        fi
                        echo "listing files under /mnt/models ..."
                        ls /mnt/models
                        echo "listing dataset files ..."
                        ls ${DATASET_NAME}
                      env:
                      # Step 1: set DATASET_URL to download dataset from gitlab.
                      - name: DATASET_URL
                        value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amldatasets/identity-alauda"
                      # Step 2: set GIT_USER and GIT_TOKEN to access private git repo.
                      # NOTE: if your dataset is located in different storage like S3, you need to modify the initializer container to download dataset from S3 instead of git.
                      - name: GIT_USER
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_USER
                      - name: GIT_TOKEN
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_TOKEN
                      resources:
                        requests:
                          cpu: 100m
                          memory: 128Mi
                        limits:
                          cpu: 2
                          memory: 4Gi
                      securityContext:
                        allowPrivilegeEscalation: false
                        capabilities:
                          drop:
                            - ALL
                        runAsNonRoot: true
                        seccompProfile:
                          type: RuntimeDefault
        - name: model-initializer
          dependsOn:
            - name: dataset-initializer
              status: Complete
          template:
            metadata:
              labels:
                trainer.kubeflow.org/trainjob-ancestor-step: model-initializer
            spec:
              template:
                spec:
                  securityContext:
                    runAsNonRoot: true
                    runAsUser: 65534
                    runAsGroup: 65534
                    fsGroup: 65534
                  containers:
                    - name: model-initializer
                      image: alaudadockerhub/fine_tune_with_llamafactory:v0.1.11
                      command:
                      - /bin/bash
                      - -c
                      - |
                        set -ex
                        cd /mnt/models
                        BASE_MODEL_NAME=$(basename ${BASE_MODEL_URL})
                        # Download base model
                        gitauth="${GIT_USER}:${GIT_TOKEN}"
                        BASE_MODEL_URL_NO_HTTPS="${BASE_MODEL_URL//https:\/\/}"
                        if [ -d ${BASE_MODEL_NAME} ]; then
                            echo "${BASE_MODEL_NAME} dir already exists, skip downloading"
                        else
                            GIT_LFS_SKIP_SMUDGE=1 git -c http.sslVerify=false -c lfs.activitytimeout=36000 clone "https://${gitauth}@${BASE_MODEL_URL_NO_HTTPS}"
                            (cd ${BASE_MODEL_NAME} && git -c http.sslVerify=false -c lfs.activitytimeout=36000 lfs pull)
                        fi
                        echo "listing files under /mnt/models ..."
                        ls /mnt/models
                        echo "listing model files ..."
                        ls ${BASE_MODEL_NAME}
                      env:
                      # Step 3: set BASE_MODEL_URL to download base model from gitlab. Make sure the GIT_USER and GIT_TOKEN have access to this git repo.
                      # NOTE: model repo name should not be the same as dataset repo name, otherwise the initializer may fail to download model and dataset correctly since they use the same PVC and the same git clone command.
                      - name: BASE_MODEL_URL
                        value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/qwen3-0.6b"
                      - name: GIT_USER
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_USER
                      - name: GIT_TOKEN
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_TOKEN
                      resources:
                        requests:
                          cpu: 100m
                          memory: 128Mi
                        limits:
                          cpu: 2
                          memory: 4Gi
                      securityContext:
                        allowPrivilegeEscalation: false
                        capabilities:
                          drop:
                            - ALL
                        runAsNonRoot: true
                        seccompProfile:
                          type: RuntimeDefault
        - name: trainer
          dependsOn:
            - name: model-initializer
              status: Complete
          template:
            metadata:
              labels:
                trainer.kubeflow.org/trainjob-ancestor-step: trainer
            spec:
              backoffLimit: 0
              template:
                spec:
                  securityContext:
                    runAsNonRoot: true
                    runAsUser: 65534
                    runAsGroup: 65534
                    fsGroup: 65534
                  volumes:
                  - name: workspace
                    emptyDir: {}
                  - name: dshm
                    emptyDir:
                      medium: Memory
                      # Step 4: set sizeLimit for dshm volume to tune the performance of multi GPU training.
                      sizeLimit: 2Gi
                  containers:
                    - name: node
                      image: alaudadockerhub/fine_tune_with_llamafactory:v0.1.11
                      env:
                      - name: BASE_MODEL_URL
                        value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/qwen3-0.6b"
                      - name: DATASET_URL
                        value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amldatasets/identity-alauda"
                      - name: GIT_USER
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_USER
                      - name: GIT_TOKEN
                        valueFrom:
                          secretKeyRef:
                            name: aml-image-builder-secret
                            key: MODEL_REPO_GIT_TOKEN
                      - name: HF_HOME
                        value: /mnt/workspace/hf_cache
                      - name: DO_MERGE
                        value: "true"
                      - name: MLFLOW_TRACKING_URI
                        value: "http://mlflow-tracking-server.kubeflow:5000"
                      - name: MLFLOW_EXPERIMENT_NAME
                        value: mlops-demo-ai-test
                      - name: MODEL_OUTPUT_DIR
                        value: /mnt/workspace/output_model
                      - name: OUTPUT_MODEL_URL
                        value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/wy-sft-output"
                      command:
                        - bash
                        - -c
                        - |
                          set -ex
                          echo "job workers list: ${VC_WORKER_HOSTS}"
                          if [ "${VC_WORKER_HOSTS}" != "" ]; then
                              export N_RANKS=$(echo "${VC_WORKER_HOSTS}" |awk -F',' '{print NF}')
                              export RANK=$VC_TASK_INDEX
                              export MASTER_HOST=$(echo "${VC_WORKER_HOSTS}" |awk -F',' '{print $1}')
                              export RANK=$RANK
                              export WORLD_SIZE=$N_RANKS
                              export NNODES=$N_RANKS
                              export NODE_RANK=$RANK
                              export MASTER_ADDR=${MASTER_HOST}
                              export MASTER_PORT="8888"
                          else
                              export N_RANKS=1
                              export RANK=0
                              export NNODES=1
                              export MASTER_HOST=""
                          fi

                          cd /mnt/workspace
                          BASE_MODEL_NAME=$(basename ${BASE_MODEL_URL})
                          DATASET_NAME=$(basename ${DATASET_URL})

                          cat >lf-sft.yaml <<EOL
                          model_name_or_path: /mnt/models/${BASE_MODEL_NAME}

                          stage: sft
                          do_train: true
                          finetuning_type: lora
                          lora_target: all
                          lora_rank: 8
                          lora_alpha: 16
                          lora_dropout: 0.1

                          dataset: identity_alauda
                          dataset_dir: /mnt/models/${DATASET_NAME}
                          template: qwen
                          cutoff_len: 1024
                          max_samples: 1000
                          overwrite_cache: true
                          preprocessing_num_workers: 8

                          output_dir: output_models
                          logging_steps: 10
                          save_steps: 500
                          plot_loss: true
                          overwrite_output_dir: true

                          # global batch size: 8
                          per_device_train_batch_size: 2
                          gradient_accumulation_steps: 2
                          learning_rate: 2.0e-4
                          num_train_epochs: 4.0
                          bf16: false
                          fp16: true
                          ddp_timeout: 180000000

                          val_size: 0.1
                          per_device_eval_batch_size: 1
                          eval_strategy: steps
                          eval_steps: 500
                          report_to: mlflow
                          EOL

                          # Run training
                          if [ ${NNODES} -gt 1 ]; then
                              echo "deepspeed: ds-z3-config.json" >> lf-sft.yaml
                              FORCE_TORCHRUN=1 llamafactory-cli train lf-sft.yaml
                          else
                              unset NNODES
                              unset NODE_RANK
                              unset MASTER_ADDR
                              unset MASTER_PORT
                              llamafactory-cli train lf-sft.yaml
                          fi

                          if [ "${DO_MERGE}" == "true" ]; then
                            # Merge LoRA adapters
                            cat >lf-merge-config.yaml <<EOL
                          model_name_or_path: /mnt/models/${BASE_MODEL_NAME}
                          adapter_name_or_path: output_models
                          template: qwen
                          finetuning_type: lora

                          ### export
                          export_dir: output_models_merged
                          export_size: 4
                          export_device: cpu
                          export_legacy_format: false
                          EOL
                                
                            llamafactory-cli export lf-merge-config.yaml
                          else
                            # move output adapter for push
                            mv output_models output_models_merged
                          fi

                          # push merged model to model repo
                          gitauth="${GIT_USER}:${GIT_TOKEN}"
                          cd /mnt/workspace/output_models_merged
                          touch README.md
                          OUTPUT_MODEL_NO_HTTPS="${OUTPUT_MODEL_URL//https:\/\/}"
                          PUSH_URL="https://${gitauth}@${OUTPUT_MODEL_NO_HTTPS}"
                          push_branch=$(date +'%Y%m%d-%H%M%S')

                          git init
                          git checkout -b sft-${push_branch}
                          git lfs track *.safetensors
                          git add .
                          git -c user.name='AMLSystemUser' -c user.email='aml_admin@cpaas.io' commit -am "fine tune push auto commit"
                          git -c http.sslVerify=false -c lfs.activitytimeout=36000 push -u ${PUSH_URL} sft-${push_branch}
                      securityContext:
                        allowPrivilegeEscalation: false
                        capabilities:
                          drop:
                            - ALL
                        runAsNonRoot: true
                        seccompProfile:
                          type: RuntimeDefault
                      volumeMounts:
                        - name: workspace
                          mountPath: /mnt/workspace
                        - name: dshm
                          mountPath: /dev/shm


Follow above comments to configure the defaults of your template. You may also need to check the secret `aml-image-builder-secret` exists under your namespace, create your own secret if it does not exist.

Apply the `TrainingRuntime` to your namespace:

### Optional: Use Huawei Ascend NPU

If you want to run the same fine-tuning workflow on Huawei Ascend NPU, create a dedicated NPU `TrainingRuntime` instead of reusing the default GPU-oriented runtime.

- Use the pre-built image `alaudadockerhub/fine_tune_with_llamafactory_npu:v0.9.4-cann_8.5.0-torch_2.6.0-v1`, or build your own image from `fine_tune_with_llamafactory_npu.Containerfile`.
- In the NPU `TrainingRuntime`, replace the image for `dataset-initializer`, `model-initializer`, and `trainer`.
- Set the pod `securityContext` of each replicated job to:

```yaml
securityContext:
  runAsNonRoot: true
  runAsUser: 1001
  runAsGroup: 1000
  fsGroup: 1000
```

These values are required for Ascend NPU workloads running on Kubernetes with the device plugin.

When you submit the `TrainJob`, request Ascend resources from the device plugin in `spec.trainer.resourcesPerNode`, for example:

```yaml
resourcesPerNode:
  limits:
    cpu: "4"
    memory: "16Gi"
    huawei.com/Ascend910: "1"
  requests:
    cpu: "100m"
    memory: "2Gi"
    huawei.com/Ascend910: "1"
```

Use the exact Ascend resource key exposed by your cluster's device plugin if it differs from `huawei.com/Ascend910`.

After creating the NPU runtime, update `TrainJob.spec.runtimeRef.name` to point to the NPU runtime, and set node selectors and accelerator resources according to your cluster's Ascend device plugin configuration.

#### Ascend Distributed Training Notes

For multi-node Ascend jobs, add a few extra constraints beyond the generic GPU example:

- Use `spec.trainer.numNodes > 1` only when each trainer replica runs on a different Kubernetes node. If you want to use two NPUs from the same host, keep `numNodes: 1` and expose two Ascend devices to a single trainer pod.
- Use network storage for the shared model and dataset PVC.
- Add `podAntiAffinity` or `topologySpreadConstraints` on `kubernetes.io/hostname` so the trainer replicas do not collapse onto the same host.
- Set `hostNetwork: true` and `dnsPolicy: ClusterFirstWithHostNet` on the trainer replicated job for HCCL-based cross-node communication.
- Set `HCCL_SOCKET_IFNAME` to the real host NIC used for inter-node Ascend traffic, for example `=enp67s0f0np0`. Do not assume `status.hostIP` is the correct HCCL address. Use `HCCL_IF_IP` only if you can inject a different local IP for each trainer pod.
- If logs show `507033` or `NN_Process_Bin_Error(E30004)`, investigate the Ascend device state on the node (`npu-smi`, Ascend logs, ECC or device faults). That is a device-side problem, not a `TrainJob` YAML problem.

Use the replicated training job name from your runtime template in `targetJobs`. The example below uses `trainer`, which matches the runtime in this notebook:

```yaml
podTemplateOverrides:
  - targetJobs:
      - name: trainer
    spec:
      hostNetwork: true
      dnsPolicy: ClusterFirstWithHostNet
      affinity:
        podAntiAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            - topologyKey: kubernetes.io/hostname
              labelSelector:
                matchLabels:
                  trainer.kubeflow.org/trainjob-ancestor-step: trainer
trainer:
  numNodes: 2
  env:
    - name: HCCL_SOCKET_IFNAME
      value: "=enp67s0f0np0"
    - name: HCCL_SOCKET_FAMILY
      value: AF_INET
```


In [ ]:
# Replace <namespace> with your target namespace, e.g., kubeflow or your team namespace
kubectl apply -f kf-trainingruntime.yaml -n <namespace>

In [ ]:
# Verify the TrainingRuntime was created successfully
kubectl get trainingruntime llamafactory-finetune-runtime -n <namespace>

## Step 2: Submit a TrainJob

With the `TrainingRuntime` in place, you can now submit a `TrainJob` for each fine-tuning experiment. The `TrainJob` only specifies:

- **Which runtime to use** (`spec.runtimeRef`)
- **Volume/node overrides** (`spec.podTemplateOverrides`) — mounts the shared PVC and sets the node selectors 
- **Per-run environment variables** (`spec.initializer` and `spec.trainer.env`) — model URL, dataset URL, output path, etc.
- **Resource requests** (`spec.trainer.resourcesPerNode`) — CPU, memory, GPU allocation

> **Tip:** Use `generateName` instead of a fixed `name` so that each submission automatically gets a unique name — this lets you track separate experiment runs easily.

Save the following as `kf-trainjob.yaml`:

In [ ]:
%%writefile kf-trainjob.yaml
apiVersion: trainer.kubeflow.org/v1alpha1
kind: TrainJob
metadata:
  generateName: trainjob-sft-qwen3-
  namespace: mlops-demo-ai-test
  labels:
    kueue.x-k8s.io/queue-name: local-queue
spec:
  runtimeRef:
    apiGroup: trainer.kubeflow.org
    name: llamafactory-finetune-runtime
    kind: TrainingRuntime
  podTemplateOverrides:
    - targetJobs:
        - name: trainer
      spec:
        # Step 1: Configure models-cache volume override to mount the shared PVC for caching models
        # In distributed training tasks (with >= 2 replicas), ensure that you use the appropriate storage type for caching large models:
        # - Network storage, such as NFS or Ceph: Simply mount the network storage. Note that multiple containers may access this network storage simultaneously, resulting in high concurrent traffic. Furthermore, reading large model files may be slower than reading them locally (depending on the network storage's performance).
        # - Local storage, such as topolvm or local-storage: Use `kserve local model cache` to pre-cache the model file on each node before mounting this PVC. Training tasks cannot cache each local PVC.
        volumes:
          - name: models-cache
            persistentVolumeClaim:
              claimName: team-model-cache-pvc
        containers:
          - name: node
            volumeMounts:
              - name: models-cache
                mountPath: /mnt/models
        # Optional: for multi-node distributed training, keep trainer replicas on
        # different Kubernetes nodes with pod anti-affinity or topology spread.
        # affinity:
        #   podAntiAffinity:
        #     requiredDuringSchedulingIgnoredDuringExecution:
        #       - topologyKey: kubernetes.io/hostname
        #         labelSelector:
        #           matchLabels:
        #             trainer.kubeflow.org/trainjob-ancestor-step: trainer
        # Step 2: Configure node selector to ensure the job runs on GPU nodes if needed.
        nodeSelector:
          nvidia.com/gpu.product: Tesla-T4
    - targetJobs:
        - name: dataset-initializer
      spec:
        # Step 3: Do the same as step 1.
        volumes:
          - name: models-cache
            persistentVolumeClaim:
              claimName: team-model-cache-pvc
        containers:
          - name: dataset-initializer
            volumeMounts:
              - name: models-cache
                mountPath: /mnt/models
        nodeSelector:
          nvidia.com/gpu.product: Tesla-T4
    - targetJobs:
        - name: model-initializer
      spec:
        # Step 4: Do the same as step 1.
        volumes:
          - name: models-cache
            persistentVolumeClaim:
              claimName: team-model-cache-pvc
        containers:
          - name: model-initializer
            volumeMounts:
              - name: models-cache
                mountPath: /mnt/models
        nodeSelector:
          nvidia.com/gpu.product: Tesla-T4
  initializer:
    # Step 5: set dataset and model URL in initializer step. The initializer will download dataset and model to the shared PVC, so that the trainer can access them from the same path.
    dataset:
      env:
        - name: DATASET_URL
          value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amldatasets/identity-alauda"
    model:
      env:
        - name: BASE_MODEL_URL
          value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/qwen3-0.6b"
  trainer:
    # Step 6: set numNodes to fit the parallelism requirement of your training job.
    # Use `numNodes: 2+` only when trainer replicas run on different Kubernetes
    # nodes. If you want two accelerators from the same host, keep `numNodes: 1`
    # and allocate both devices to a single trainer pod.
    numNodes: 1
    env:
    # Step 7: set environments accordingly.
    - name: BASE_MODEL_URL
      value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/qwen3-0.6b"
    - name: DATASET_URL
      value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amldatasets/identity-alauda"
    # For distributed Ascend NPU jobs with host networking, also set HCCL
    # networking variables, for example:
    # - name: HCCL_SOCKET_IFNAME
    #   value: "=enp67s0f0np0"
    # - name: HCCL_SOCKET_FAMILY
    #   value: AF_INET
    - name: HF_HOME
      value: /mnt/workspace/hf_cache
    - name: DO_MERGE
      value: "true"
    - name: MLFLOW_TRACKING_URI
      value: "http://mlflow-tracking-server.kubeflow:5000"
    - name: MLFLOW_EXPERIMENT_NAME
      value: mlops-demo-ai-test
    - name: MODEL_OUTPUT_DIR
      value: /mnt/workspace/output_models_merged
    - name: OUTPUT_MODEL_URL
      value: "https://aml-gitlab.alaudatech.net/mlops-demo-ai-test/amlmodels/wy-sft-output"
    resourcesPerNode:
      limits:
        cpu: "4"
        memory: "16Gi"
        nvidia.com/gpualloc: 1
        nvidia.com/gpucores: 50
        nvidia.com/gpumem: "8192"
      requests:
        cpu: "100m"
        memory: "2Gi"
        nvidia.com/gpualloc: 1
        nvidia.com/gpucores: 50
        nvidia.com/gpumem: "8192"


### Key Fields in TrainJob

| Field | Purpose |
|---|---|
| `metadata.labels[kueue.x-k8s.io/queue-name]` | Name of the `LocalQueue` to submit this job to (requires Kueue; see Step 2b) |
| `spec.runtimeRef.name` | Must match the `TrainingRuntime` name created in Step 1 |
| `spec.podTemplateOverrides` | Injects volume mounts, node selectors, and scheduling or network overrides into each `replicatedJob` without modifying the runtime template |
| `spec.initializer.dataset.env` / `spec.initializer.model.env` | Override `DATASET_URL` / `BASE_MODEL_URL` for the initializer steps of this specific run |
| `spec.trainer.numNodes` | Set to `1` for single-node training. Set to `2+` only when each trainer replica runs on a different Kubernetes node for real multi-node training. |
| `spec.trainer.env` | Override any environment variables in the trainer container (model URL, dataset URL, output model URL, HCCL networking variables for Ascend, etc.) |
| `spec.trainer.resourcesPerNode` | CPU, memory, and accelerator requests and limits **per training node** |

Submit the `TrainJob`:


In [ ]:
# Submit the TrainJob. Use 'create' (not 'apply') because generateName is used.
kubectl create -f kf-trainjob.yaml

## Step 2b: (Optional) Scheduling with Kueue

[Kueue](https://kueue.sigs.k8s.io/) provides job queuing, quota management, and fair scheduling for Kubernetes workloads. When Kueue is installed in your cluster, TrainJobs are held in a **suspended** state until Kueue admits them based on available quota.

### How it works

1. A cluster admin creates a `ClusterQueue` with resource quotas (CPU, memory, GPU).
2. A namespace admin creates a `LocalQueue` pointing to the `ClusterQueue`.
3. Users label their `TrainJob` with `kueue.x-k8s.io/queue-name` to submit it to a `LocalQueue`.
4. Kueue evaluates the resource request, admits the workload when quota is available, and unsuspends the job.

### Create a LocalQueue

Before submitting TrainJobs with Kueue, create a `LocalQueue` in your namespace that references an existing `ClusterQueue`:

In [ ]:
%%writefile kf-local-queue.yaml
apiVersion: kueue.x-k8s.io/v1beta2
kind: LocalQueue
metadata:
  name: local-queue
  namespace: mlops-demo-ai-test
spec:
  clusterQueue: cluster-queue

In [ ]:
kubectl apply -f kf-local-queue.yaml

In [ ]:
# Verify the LocalQueue was created and is pointing to the ClusterQueue
kubectl get localqueue -n mlops-demo-ai-test

### Kueue label in TrainJob

The `kf-trainjob.yaml` created above already includes the Kueue label:

```yaml
metadata:
  labels:
    kueue.x-k8s.io/queue-name: local-queue
```

When this label is present:
- The TrainJob starts in `Suspended` state.
- Kueue's `ClusterQueue` evaluates whether there is enough quota to run the job.
- Once admitted, Kueue unsuspends the job and pods begin scheduling.

> **Note:** If Kueue is **not** installed in your cluster, simply remove the `kueue.x-k8s.io/queue-name` label from the TrainJob metadata — the job will run immediately without queue-based scheduling.

### PodsReady timeout

The cluster may have a Kueue **PodsReady timeout** (e.g., 5 minutes). If the training container image is large and not yet cached on the target node, the first job attempt may be evicted with a `PodsReadyTimeout` reason. Resubmitting usually succeeds because the image is now cached locally.

### Ascend NPU distributed training notes

When using Huawei Ascend for distributed training:

- `spec.trainer.numNodes: 2` means two trainer replicas on two different Kubernetes nodes. If both replicas may land on the same host, add anti-affinity or topology spread on `kubernetes.io/hostname`.
- If you intentionally want two NPUs from the same host, use `numNodes: 1` and request both NPUs for one trainer pod instead of creating two 1-NPU pods.
- When `hostNetwork: true` is enabled for HCCL, set `HCCL_SOCKET_IFNAME` to the actual inter-node NIC, for example `=enp67s0f0np0`. Do not blindly use `status.hostIP` as the HCCL address.
- If the job fails with `507033` or `NN_Process_Bin_Error(E30004)`, check `npu-smi` and the Ascend device logs on the target node. That indicates a device fault rather than a `TrainJob` spec problem.

You can check workload admission status with:


In [ ]:
# Check Kueue workload status for the namespace
kubectl get workloads -n mlops-demo-ai-test -o wide

## Step 3: Monitor the Training Job

### List All TrainJobs

Use `kubectl` to list all `TrainJob` resources in your namespace and check their status:

In [ ]:
# List all TrainJobs in the namespace
kubectl get trainjobs -n <namespace>

### Inspect a Specific TrainJob

Replace `<trainjob-name>` with the actual name returned above (e.g., `trainjob-sft-qwen3-abc12`):

In [ ]:
# Show the detailed status of a specific TrainJob
kubectl describe trainjob <trainjob-name> -n <namespace>

### View Pod Logs

The Trainer v2 creates pods for each step of the pipeline. Pod names follow the pattern `<trainjob-name>-<step>-0-0`. Check the pods created by the job:

In [ ]:
# List all pods created by the TrainJob (they share the job name as a label prefix)
kubectl get pods -n <namespace> -l trainer.kubeflow.org/trainjob-name=<trainjob-name>

In [ ]:
# Stream logs from the dataset-initializer step
kubectl logs -f -n <namespace> -l trainer.kubeflow.org/trainjob-name=<trainjob-name>,trainer.kubeflow.org/trainjob-ancestor-step=dataset-initializer

In [ ]:
# Stream training logs from the trainer step (shows LlamaFactory progress)
kubectl logs -f -n <namespace> -l trainer.kubeflow.org/trainjob-name=<trainjob-name>,trainer.kubeflow.org/trainjob-ancestor-step=trainer

## Step 4: Run Multiple Experiments

Because the `TrainingRuntime` is a reusable template, you can run multiple fine-tuning experiments simultaneously (or sequentially) by submitting different `TrainJob`s that override just the parts you want to change.


> **Note:** Model and dataset repo names **must be different** from each other. The initializer containers clone them into the same directory path (`/mnt/models/<repo-name>`), so identical names would cause one to overwrite the other.

## Step 5: View Training Metrics in MLflow

If `MLFLOW_TRACKING_URI` is set and the MLflow server is reachable from the training pod, LlamaFactory will log metrics (loss, learning rate, etc.) to MLflow automatically via `report_to: mlflow` in the training config.

To open the MLflow UI, go to **Alauda AI** - **Tools** - **MLFlow** (need MLFlow Cluster plugin installed). Look for the experiment named by `MLFLOW_EXPERIMENT_NAME`.

Each `TrainJob` run will appear as a separate MLflow **run** under the same experiment, making it easy to compare training curves across different models and hyperparameters.

## Step 6: Launch Inference Service Using the Fine-tuned Model

After the fine-tuning task completes, the model is automatically pushed to the model repository. You can use the fine-tuned model to launch the inference service and access it.

> **Note:** In the example task above, we use LoRA fine-tuning method. Before uploading the model, the LoRA adapter was merged with the original model. This allows the output model to be directly published to the inference service. ***Launching inference service using base model and adapters are NOT supported in current version.***

Steps:

1. Go to **Alauda AI** - **Model Repository**, find the fine-tuned output model, go to **Model Info** - **File Management** - **Edit Metadata**, select "Task Type": "Text Generation", and "Framework": "Transformers".
2. Click the **Publish Inference API** button, then select **Custom Publishing**.
3. On the Publish Inference Service page, select **vLLM** inference runtime (select vLLM with the CUDA version that is supported by the cluster GPU nodes), fill in settings of storage, resource, GPU, then click **Publish**.
4. Wait until the inference service is fully started, click the **Experience** button in the upper-right corner to start a conversation with the model. (Note: Models that include the `chat_template` configuration only have chat capabilities.)

## Step 7: Cleanup

Delete individual `TrainJob` resources once they are complete and no longer needed:

In [ ]:
# Delete a specific completed TrainJob and its associated pods
kubectl delete trainjob <trainjob-name> -n <namespace>

In [ ]:
# Delete all TrainJobs in the namespace (use with caution)
kubectl delete trainjobs --all -n <namespace>

# If you want to remove the TrainingRuntime template as well
kubectl delete trainingruntime llamafactory-finetune-runtime -n <namespace>

## Summary

| Step | Resource | Action |
|---|---|---|
| 1 | `TrainingRuntime` | Create **once** — defines the pipeline image, steps, and default configuration |
| 2 | `TrainJob` | Create **per experiment** — specifies model, dataset, resources, and output |
| 2b | `LocalQueue` | (Optional) Create once per namespace — enables Kueue scheduling and quota management |
| 3 | Pods | Monitor logs via `kubectl logs` using step labels |
| 4–6 | MLflow / Git | Track metrics in MLflow; retrieve merged model from the output Git branch |

### Common Customizations

| What to change | Where to change it |
|---|---|
| Base model URL | `spec.initializer.model.env[DATASET_URL]` and `spec.trainer.env[BASE_MODEL_URL]` in `TrainJob` |
| Dataset URL | `spec.initializer.dataset.env[DATASET_URL]` and `spec.trainer.env[DATASET_URL]` in `TrainJob` |
| Number of training epochs | Modify `num_train_epochs` inside the `cat >lf-sft.yaml` block in `TrainingRuntime` |
| LoRA rank / alpha | Modify `lora_rank` / `lora_alpha` in `TrainingRuntime` |
| GPU count per node | `spec.trainer.resourcesPerNode.limits.nvidia.com/gpualloc` in `TrainJob` |
| Use Huawei Ascend NPU | Create a dedicated `TrainingRuntime`, set `runAsUser: 1001`, `runAsGroup: 1000`, `fsGroup: 1000`, and request Ascend resources such as `huawei.com/Ascend910: "1"` in `spec.trainer.resourcesPerNode` |
| Ascend HCCL networking | For distributed Ascend jobs, set `hostNetwork: true`, `dnsPolicy: ClusterFirstWithHostNet`, and `HCCL_SOCKET_IFNAME` to the real host NIC used for inter-node communication |
| Multi-node training | Set `spec.trainer.numNodes` > 1, use network storage for the PVC, and keep trainer replicas on different Kubernetes nodes with anti-affinity or topology spread |
| Two accelerators on one host | Keep `spec.trainer.numNodes: 1` and allocate both devices in `spec.trainer.resourcesPerNode` for a single trainer pod |
| Enable Kueue scheduling | Add `kueue.x-k8s.io/queue-name: <local-queue-name>` label to `TrainJob` metadata |
| Skip LoRA merge | Set `DO_MERGE=false` in `spec.trainer.env` in `TrainJob` |
| Output model repository | Set `OUTPUT_MODEL_URL` in `spec.trainer.env` in `TrainJob` |
